# PANDAPROSUMER EXAMPLE: Energy System

## DESCRIPTION:
This example demonstrates how to create a single supervisor controller in pandaprosumer and use it to manage the parameters of other controllers within a prosumer. In this example, the supervisor controls a heat pump and a gas boiler.
 If the gas price is very low, the gas boiler is prioritized. If the gas price is not low, the heat pump is prioritized. If the gas price is extremely high, the gas boiler is turned off—even if the heat pump cannot meet the full demand. The demand and source temperature data is read from an Excel file and stored in pandas dataframe. It includes the information about the power required by the consumer and source temperature at each time step.


![title](img/energy_system.png)

## Glossary:
- Network: A configuration of connected energy generators and energy consumers
- Element: A single energy generator or a single energy consumer
- Controller: The logic of an element that defines its behaviour and its limits
- Prosumer/Container: A pandaprosumer data structure that holds data related to elements and their controllers.
- Const Profile Controller: The initial controller in the network that interacts with other element controllers; it also manages external data via time series.
- Supervisor: A controller that can modify the parameters of other controllers and elements within the prosumer during the calculation process.
- Map / mapping: A connection between two controllers that specifies what information is exchanged between the corresponding elements.

## Network design philosophy:
In pandaprosumer, a system's component is represented by a network element. Each element is assigned a container and its own element controller. A container is a structure that contains the component's configuration data (static input data), which can include information that will not change in the analysis such as size, nominal power, efficiency, etc. The behaviour of an element is governed by its controller. Connections between elements are defined in maps, which couple output parameters of one controller to the input parameter of a controller of a connected element. The network is managed by a controller called ConstProfileController. This controller is connected to all element controllers and manages dynamic input data from external sources (e.g. Excel file). For each time step it distributes the dynamic input data to the relevant element controllers.

# 1 - INPUT DATA:
First let's import libraries required for data management.

In [ ]:
import pandapower
from pandapower.timeseries import OutputWriter

from pandaprosumer.create_controlled import *
from pandaprosumer.energy_system import create_empty_energy_system, add_net_to_energy_system, \
    add_pandaprosumer_to_energy_system
from pandaprosumer.energy_system.control.controller.coupling.network_coupling import NetworkCouplingControl
from pandaprosumer.energy_system.control.controller.data_model.network_coupling import NetworkCouplingData
from pandaprosumer.energy_system.timeseries.run_time_series_energy_system import \
    run_timeseries as run_time_series_system
from pandaprosumer.mapping import GenericMapping, FluidMixMapping, FluidMixEnergySystemMapping, \
    GenericEnergySystemMapping

We define the analysis time series.

In [ ]:
start = '2020-01-01 00:00:00'
time_resolution_s = 1

Now we import our demand data and transform it into an appropriate DFData object. All data of an individual element is stored in a dedicated DFData object.

In [ ]:
import sys
import os

current_directory = os.getcwd()
parent_directory = os.path.dirname(current_directory)
sys.path.append(parent_directory)

demand_data = pd.read_excel('data/energy_system.xlsx')

end = pd.Timestamp(start) + len(demand_data) * pd.Timedelta(f"00:00:{time_resolution_s}") - pd.Timedelta("00:00:01")

dur = pd.date_range(start=start, end=end, freq=f'{time_resolution_s}s', tz='utc')
demand_data.index = dur
demand_input = DFData(demand_data)
print(demand_input.df.head(10))

We can plot the evolution of the demand from the Excel file.

In [ ]:
import matplotlib.pyplot as plt

demand_input.df.plot(y='Tin,evap')
plt.title("External temperature")
plt.ylabel("Temperature (°C)")
plt.show()

In [ ]:
demand_input.df.plot(y='demand_1')
plt.title("Heat Consumer demand power")
plt.ylabel("Thermal power (kW)")
plt.show()

# 2 - CREATING ELEMENTS OF THE NETWORK:
In this example, the network is made up of four elements: a supervisor, two source and a consumer. The source is represented by a heat pump or a gas boiler and the consumer is modelled by a single heat demand element.

We begin by defining an empty prosumer container object and then add the different elements and their respective controllers to it.

In [ ]:
def _create_pipes_network(time_resolution_s):
    net = pandapipes.create_empty_network(fluid="water", name='net_pipes')
    t_amb_k = 293
    pandapipes.set_user_pf_options(net, ambient_temperature=t_amb_k, mode='bidirectional')

    # Create junctions
    j0 = pandapipes.create_junction(net, pn_bar=10, tfluid_k=350, geodata=(0, 1))
    j1 = pandapipes.create_junction(net, pn_bar=10, tfluid_k=350, geodata=(2, 1))
    j2 = pandapipes.create_junction(net, pn_bar=10, tfluid_k=350, geodata=(2, 0))
    j3 = pandapipes.create_junction(net, pn_bar=10, tfluid_k=350, geodata=(0, 0))

    # Create branched elements
    pandapipes.create_pipes_from_parameters(net, from_junctions=[j0, j2], to_junctions=[j1, j3], length_km=1,
                                            diameter_m=0.05, u_w_per_m2k=10, text_k=t_amb_k)

    pandapipes.create_circ_pump_const_pressure(net,
                                               j3,
                                               j0,
                                               p_flow_bar=10,
                                               plift_bar=5,
                                               t_flow_k=350)

    pandapipes.create_heat_consumer(net, from_junction=j1, to_junction=j2,
                                    qext_w=100e3, controlled_mdot_kg_per_s=3)

    pandapipes.set_user_pf_options(
        net,
        mode='bidirectional',
        transient=True, dt=time_resolution_s)

    return net

In [ ]:
def _create_power_network():
    net = pandapower.create_empty_network(name='net_power')
    b1 = pandapower.create_bus(net, vn_kv=20.)
    b2 = pandapower.create_bus(net, vn_kv=20.)
    pandapower.create_line(net, from_bus=b1, to_bus=b2, length_km=2.5, std_type="NAYY 4x50 SE")
    pandapower.create_ext_grid(net, bus=b1)
    pandapower.create_load(net, bus=b2, p_mw=1.)
    return net

In [ ]:
def _create_prosumer_prod(hp_level):
    prosumer = create_empty_prosumer_container(name='prosumer_prod', check_order=False)
    data_source = demand_input
    period = create_period(prosumer, time_resolution_s, start, end, 'utc', 'default')

    cp_input_columns = ["Tin,evap"]
    cp_result_columns = ["Tin,evap"]

    hp_params = {'carnot_efficiency': 0.5,
                 'pinch_c': 5,
                 'delta_t_evap_c': 8,
                 'max_p_comp_kw': 1000e3}

    cp_controller_index = create_controlled_const_profile(prosumer, cp_input_columns, cp_result_columns,
                                                          data_source, period, 0, 0)

    hp_controller_index = create_controlled_heat_pump(prosumer, period=period, name='hp_controller',
                                                      level=hp_level, order=0, **hp_params)
    GenericMapping(container=prosumer,
                   initiator_id=cp_controller_index,
                   initiator_column="Tin,evap",
                   responder_id=hp_controller_index,
                   responder_column="t_evap_in_c",
                   order=0)

    return prosumer

In [ ]:
def _create_prosumer_dmd(level):
    prosumer = create_empty_prosumer_container(name='prosumer_dmd', check_order=False)
    data_source = demand_input
    period = create_period(prosumer, time_resolution_s, start, end, 'utc', 'default')

    cp_input_columns = ["demand_1"]
    cp_result_columns = ["demand_kw"]

    hx_params = {'t_1_in_nom_c': 45,
                 't_1_out_nom_c': 30,
                 't_2_in_nom_c': 20,
                 't_2_out_nom_c': 40,
                 'mdot_2_nom_kg_per_s': 3.58}

    hd_params = {'t_in_set_c': 40,
                 't_out_set_c': 20}

    cp_controller_index = create_controlled_const_profile(prosumer, cp_input_columns, cp_result_columns,
                                                          data_source, period, 0, 0)

    hx_controller_index = create_controlled_heat_exchanger(prosumer, period=period, name='hx_controller',
                                                           level=level, order=0, **hx_params)
    hd_controller_index = create_controlled_heat_demand(prosumer, period=period, name='hd_controller',
                                                        level=level, order=1, **hd_params)

    GenericMapping(container=prosumer,
                   initiator_id=cp_controller_index,
                   initiator_column="demand_kw",
                   responder_id=hd_controller_index,
                   responder_column="q_demand_kw",
                   order=0)

    FluidMixMapping(container=prosumer,
                    initiator_id=hx_controller_index,
                    responder_id=hd_controller_index,
                    order=0)

    return prosumer

In [ ]:
def _create_energy_system(nets, prosumers, name="test_energy_system"):
    energy_system = create_empty_energy_system(name=name)
    sample_prosumer_period = prosumers[0].period
    create_period(energy_system, sample_prosumer_period.iloc[0]["resolution_s"],
                  sample_prosumer_period.iloc[0]["start"],
                  sample_prosumer_period.iloc[0]["end"],
                  timezone=sample_prosumer_period.iloc[0]["timezone"],
                  name=sample_prosumer_period.iloc[0]["name"])
    for net in nets:
        add_net_to_energy_system(energy_system, net, net_name=net.name)
    for prosumer in prosumers:
        add_pandaprosumer_to_energy_system(energy_system, prosumer, pandaprosumer_name=prosumer.name)
    return energy_system

In [ ]:
net = _create_pipes_network(time_resolution_s)
net_power = _create_power_network()
prosumer_prod = _create_prosumer_prod(hp_level=2)
prosumer_dmd = _create_prosumer_dmd(level=3)
energy_system = _create_energy_system([net, net_power], [prosumer_prod, prosumer_dmd])

# 4 - CREATING CONNECTIONS (MAPS) BETWEEN THE CONTROLLERS:
Network configuration

For each controller we define how it is connected to other controllers.



In [ ]:
def create_controlled_network_coupling(net,
                                       element_index,
                                       element_name='heat_consumer',
                                       input_columns=[],
                                       result_columns=[],
                                       temp_fluid_map_input_col=[],
                                       mdot_fluid_map_input_col=[],
                                       temp_fluid_map_output_idx=None,
                                       mdot_fluid_map_output_idx=None,
                                       level=0,
                                       order=0):
    if isinstance(element_index, (np.integer, int)):
        element_index = [int(element_index)]
    elif isinstance(element_index, np.ndarray):
        element_index = [int(i) for i in element_index.tolist()]
    elif isinstance(element_index, list):
        element_index = [int(i) for i in element_index]

    networkcoupling = NetworkCouplingData(element_index=element_index,
                                          element_name=element_name,
                                          input_columns=input_columns,
                                          result_columns=result_columns)

    n = NetworkCouplingControl(net,
                               networkcoupling,
                               temp_fluid_map_input_col=temp_fluid_map_input_col,
                               mdot_fluid_map_input_col=mdot_fluid_map_input_col,
                               temp_fluid_map_output_idx=temp_fluid_map_output_idx,
                               mdot_fluid_map_output_idx=mdot_fluid_map_output_idx,
                               level=level, order=order)

    return n.index

In [ ]:
sample_prosumer_period = prosumer_prod.period
ow_time_steps = pd.date_range(sample_prosumer_period.iloc[0]["start"], sample_prosumer_period.iloc[0]["end"],
                              freq='%ss' % int(sample_prosumer_period.iloc[0]["resolution_s"]),
                              tz=sample_prosumer_period.iloc[0]["timezone"])
OutputWriter(net, ow_time_steps, log_variables=[
    ('res_circ_pump_pressure', 't_from_k'),
    ('res_circ_pump_pressure', 't_to_k'),
    ('res_circ_pump_pressure', 'mdot_from_kg_per_s'),
    ('res_heat_consumer', 't_from_k'),
    ('res_heat_consumer', 't_to_k'),
    ('res_heat_consumer', 'mdot_from_kg_per_s'),
    ('heat_consumer', 'qext_w'),
    ('res_pipe', 'v_mean_m_per_s')
])
OutputWriter(net_power, ow_time_steps, log_variables=[('res_load', 'p_mw')])


## Pipes net coupling

In [ ]:
pump_elmt_index = 0
consumer_elmt_index = 0

In [ ]:
nc_write_pump_index = create_controlled_network_coupling(net,
                                                         pump_elmt_index,
                                                         element_name='circ_pump_pressure',
                                                         temp_fluid_map_input_col=['t_flow_k'],
                                                         mdot_fluid_map_input_col=['mdot_flow_kg_per_s'],
                                                         # FixMe: doesn't exist in circ_pump_pressure
                                                         level=4,
                                                         order=0)

nc_read_dmd_index = create_controlled_network_coupling(net,
                                                       consumer_elmt_index,
                                                       element_name='heat_consumer',
                                                       result_columns=['t_from_k', 't_to_k',
                                                                       'mdot_from_kg_per_s'],
                                                       temp_fluid_map_output_idx=0,
                                                       mdot_fluid_map_output_idx=2,
                                                       level=1,
                                                       order=2)

nc_write_dmd_index = create_controlled_network_coupling(net,
                                                        consumer_elmt_index,
                                                        element_name='heat_consumer',
                                                        input_columns=['qext_w', 'controlled_mdot_kg_per_s'],
                                                        level=4,
                                                        order=2)

In [ ]:
hp_ctrl_index = 1
hx_ctrl_index = 1

In [ ]:
FluidMixEnergySystemMapping(container=prosumer_prod,
                            initiator_id=hp_ctrl_index,
                            responder_net=net,
                            responder_id=nc_write_pump_index,
                            order=0,
                            no_chain=False)

FluidMixEnergySystemMapping(container=net,
                            initiator_id=nc_read_dmd_index,
                            responder_net=prosumer_dmd,
                            responder_id=hx_ctrl_index,
                            order=0,
                            no_chain=True)

GenericEnergySystemMapping(container=prosumer_dmd,
                           initiator_id=hx_ctrl_index,
                           initiator_column='q_exchanged_kw',
                           responder_net=net,
                           responder_id=nc_write_dmd_index,
                           responder_column='qext_w',
                           order=0,
                           conversion_function=lambda q: q * 1000)

GenericEnergySystemMapping(container=prosumer_dmd,
                           initiator_id=hx_ctrl_index,
                           initiator_column='mdot_1_kg_per_s',
                           responder_net=net,
                           responder_id=nc_write_dmd_index,
                           responder_column='controlled_mdot_kg_per_s',
                           order=1)

## Power Net coupling

In [ ]:
load_elmt_index = 0

In [ ]:
nc_write_load_index = create_controlled_network_coupling(net_power,
                                                         load_elmt_index,
                                                         element_name='load',
                                                         input_columns=['p_mw'],
                                                         level=5, order=0)

In [ ]:
GenericEnergySystemMapping(container=prosumer_prod,
                           initiator_id=hp_ctrl_index,
                           initiator_column='p_comp_kw',
                           responder_net=net_power,
                           responder_id=nc_write_load_index,
                           responder_column='p_mw',
                           order=0,
                           conversion_function=lambda p: p / 1000)

# 6 - RUNNING THE ANALYSIS:
We can now run the analysis with the input data defined above.

In [ ]:
# Run initial pipeflow to create res_ tables
# pandapipes.pipeflow(net)
# pandapower.runpp(net_power)

In [ ]:
period = 0
RESOL_S = time_resolution_s
run_time_series_system(energy_system,
                       period_index=period, continue_on_divergence=False, verbose=True,
                       transient=True, dt=RESOL_S)

# 7 - PRINTING AND PLOTTING RESULTS:
All the results of the timeseries analysis are available in the prosumer.time_series dataframe

In [ ]:
prosumer_dmd.time_series

Before plotting we have to look at the resulting dataframe to see which quantity (column) do we want to plot.

Access the results of the heat pump, gas boiler and heat demand:

In [ ]:
prosumer_dmd.time_series.data_source.loc[0].df.head(10)

In [ ]:
prosumer_dmd.time_series.data_source.loc[1].df.head(10)

In [ ]:
prosumer_prod.time_series

In [ ]:
prosumer_prod.time_series.data_source.loc[0].df.head(10)

In [ ]:
print(net.output_writer.object.loc[0].output['res_circ_pump_pressure.t_from_k'])
print(net.output_writer.object.loc[0].output['res_circ_pump_pressure.t_to_k'])
print(net.output_writer.object.loc[0].output['res_circ_pump_pressure.mdot_from_kg_per_s'])

In [ ]:
print(net.output_writer.object.loc[0].output['res_heat_consumer.t_from_k'])
print(net.output_writer.object.loc[0].output['res_heat_consumer.t_to_k'])
print(net.output_writer.object.loc[0].output['res_heat_consumer.mdot_from_kg_per_s'])

In [ ]:
print(net.output_writer.object.loc[0].output['heat_consumer.qext_w'])

In [ ]:
net_power.output_writer.object.loc[0].output['res_load.p_mw']

In [ ]:
print(net.output_writer.object.loc[0].output['heat_consumer.qext_w'])
print(prosumer_dmd.time_series.data_source.loc[0].df['q_exchanged_kw'])
print(demand_input.df['demand_1'])
print(prosumer_dmd.time_series.data_source.loc[1].df['q_received_kw'])

In [ ]:
prosumer_prod.time_series.data_source.loc[0].df['q_cond_kw']


## Comparison Graphs: Powers and Temperatures

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Power comparison
axes[0, 0].plot(prosumer_prod.time_series.data_source.loc[0].df.index,
                prosumer_prod.time_series.data_source.loc[0].df['q_cond_kw'],
                label='HP Condenser Power', linewidth=2)
axes[0, 0].plot(net.output_writer.object.loc[0].output['heat_consumer.qext_w'].index,
                net.output_writer.object.loc[0].output['heat_consumer.qext_w'] / 1000,
                label='Net Consumer Power', linewidth=2, alpha=0.7)
axes[0, 0].plot(prosumer_dmd.time_series.data_source.loc[1].df.index,
                prosumer_dmd.time_series.data_source.loc[1].df['q_received_kw'],
                label='Demand Received Power', linewidth=2, linestyle='--')
axes[0, 0].set_xlabel('Time')
axes[0, 0].set_ylabel('Power (kW)')
axes[0, 0].set_title('Power Comparison')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# HP compressor power
axes[0, 1].plot(prosumer_prod.time_series.data_source.loc[0].df.index,
                prosumer_prod.time_series.data_source.loc[0].df['p_comp_kw'],
                label='HP Compressor Power', color='red', linewidth=2)
axes[0, 1].plot(net_power.output_writer.object.loc[0].output['res_load.p_mw'].index,
                net_power.output_writer.object.loc[0].output['res_load.p_mw'] * 1000,
                label='Power Net Load', color='darkred', linewidth=2, alpha=0.7)
axes[0, 1].set_xlabel('Time')
axes[0, 1].set_ylabel('Power (kW)')
axes[0, 1].set_title('Electrical Power Consumption')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Temperature in pipes net
axes[1, 0].plot(net.output_writer.object.loc[0].output['res_circ_pump_pressure.t_from_k'].index,
                net.output_writer.object.loc[0].output['res_circ_pump_pressure.t_from_k'] - 273.15,
                label='Pump Inlet Temp', linewidth=2)
axes[1, 0].plot(net.output_writer.object.loc[0].output['res_circ_pump_pressure.t_to_k'].index,
                net.output_writer.object.loc[0].output['res_circ_pump_pressure.t_to_k'] - 273.15,
                label='Pump Outlet Temp', linewidth=2)
axes[1, 0].plot(net.output_writer.object.loc[0].output['res_heat_consumer.t_from_k'].index,
                net.output_writer.object.loc[0].output['res_heat_consumer.t_from_k'] - 273.15,
                label='Consumer Inlet Temp', linewidth=2)
axes[1, 0].plot(net.output_writer.object.loc[0].output['res_heat_consumer.t_to_k'].index,
                net.output_writer.object.loc[0].output['res_heat_consumer.t_to_k'] - 273.15,
                label='Consumer Outlet Temp', linewidth=2)
axes[1, 0].set_xlabel('Time')
axes[1, 0].set_ylabel('Temperature (°C)')
axes[1, 0].set_title('Pipe Network Temperatures')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Temperature in prosumers
axes[1, 1].plot(prosumer_prod.time_series.data_source.loc[0].df.index,
                prosumer_prod.time_series.data_source.loc[0].df['t_cond_in_c'],
                label='HP Condenser In', linewidth=2)
axes[1, 1].plot(prosumer_prod.time_series.data_source.loc[0].df.index,
                prosumer_prod.time_series.data_source.loc[0].df['t_cond_out_c'],
                label='HP Condenser Out', linewidth=2)
axes[1, 1].plot(prosumer_dmd.time_series.data_source.loc[0].df.index,
                prosumer_dmd.time_series.data_source.loc[0].df['t_1_in_c'],
                label='HX Primary In', linewidth=2, linestyle='--')
axes[1, 1].plot(prosumer_dmd.time_series.data_source.loc[0].df.index,
                prosumer_dmd.time_series.data_source.loc[0].df['t_1_out_c'],
                label='HX Primary Out', linewidth=2, linestyle='--')
axes[1, 1].set_xlabel('Time')
axes[1, 1].set_ylabel('Temperature (°C)')
axes[1, 1].set_title('Prosumer Temperatures')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
net.res_pipe


In [ ]:
net.pipe

In [ ]:
# ToDo: check water velocity

net.output_writer.object.loc[0].output['res_pipe.v_mean_m_per_s']

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Mass flow rates comparison
axes[0].plot(net.output_writer.object.loc[0].output['res_circ_pump_pressure.mdot_from_kg_per_s'].index,
             net.output_writer.object.loc[0].output['res_circ_pump_pressure.mdot_from_kg_per_s'],
             label='Net Pump Flow', linewidth=2)
axes[0].plot(net.output_writer.object.loc[0].output['res_heat_consumer.mdot_from_kg_per_s'].index,
             net.output_writer.object.loc[0].output['res_heat_consumer.mdot_from_kg_per_s'],
             label='Net Consumer Flow', linewidth=2)
axes[0].plot(prosumer_prod.time_series.data_source.loc[0].df.index,
             prosumer_prod.time_series.data_source.loc[0].df['mdot_cond_kg_per_s'],
             label='HP condenser Flow', linewidth=2, linestyle='--')
axes[0].plot(prosumer_dmd.time_series.data_source.loc[0].df.index,
             prosumer_dmd.time_series.data_source.loc[0].df['mdot_1_kg_per_s'],
             label='HX Primary Flow', linewidth=2, linestyle='--')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Mass Flow Rate (kg/s)')
axes[0].set_title('Mass Flow Rates Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# COP and efficiency
axes[1].plot(prosumer_prod.time_series.data_source.loc[0].df.index,
             prosumer_prod.time_series.data_source.loc[0].df['cop'],
             label='Heat Pump COP', linewidth=2, color='green')
axes[1].set_xlabel('Time')
axes[1].set_ylabel('COP')
axes[1].set_title('Heat Pump Coefficient of Performance')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
